# Level 4, Notebook 4: Work queues and batch reports

**Problems in this notebook**

- Problem 34: Build Queue Priority for Urgent Tickets
- Problem 40: Batch Processing Failure Summary

**How to use this notebook**

1. Read the problem and say your plan aloud before you write any code.
2. Use the AI prompt given (or your own narrow one) to fill the starter. Read the AI's code before you run it.
3. Run the checks. Open the reference only after you have tried.

The ideas behind these problems are in `04_queues_and_batch_reports_THEORY.md` in this folder.

In [ ]:
def run_checks(fn, cases):
    """cases: list of (args_tuple, expected). Prints a short report, never raises."""
    passed = 0
    for i, (args, expected) in enumerate(cases, 1):
        try:
            got = fn(*args)
        except NotImplementedError:
            print("Not implemented yet. Fill in the starter first.")
            return False
        except Exception as e:
            print(f"Case {i}: raised {type(e).__name__}: {e}")
            continue
        if got == expected:
            passed += 1
        else:
            print(f"Case {i}: expected {expected!r}, got {got!r}")
    print(f"{passed}/{len(cases)} checks passed")
    return passed == len(cases)

## Problem 34: Build Queue Priority for Urgent Tickets

**The situation**

An AI triage worker drafts replies to support tickets. Urgent and VIP tickets must be handled before normal ones, and tickets with the same priority should be handled in the order they arrived.

**Your task**

Build `TicketQueue` with `push(ticket)` and `pop()`. A lower `priority` number is more urgent. A missing priority means 5 (normal). Same priority comes out first in, first out. `pop()` on an empty queue returns `None`.

**Example**

Push `{"id": "normal", "priority": 5}`, then `{"id": "urgent", "priority": 1}`. The first `pop()` returns the urgent ticket.

**Ask before coding**

- Is a lower number more urgent, or a higher one?
- What priority does a ticket get if the field is missing?
- Can low-priority tickets wait forever if urgent ones keep arriving?

**Plan to say aloud**

1. Use `heapq` so the smallest item is always on top.
2. Push a tuple `(priority, sequence, ticket)`. The sequence number grows by one each push.
3. The sequence keeps same-priority tickets in arrival order and stops Python from comparing two dicts.
4. `pop()` returns `None` when empty, else the ticket from the top tuple.

### Prompt for your AI assistant

```text
Write a Python class TicketQueue with push(self, ticket: dict) -> None and pop(self) -> dict | None, using heapq. Priority is ticket.get('priority', 5); lower number pops first. Push (priority, sequence, ticket) where sequence is a counter that increases on every push, so equal priorities pop in arrival order. pop() returns None when the queue is empty. Show only the class.
```

**Check the AI's code for**

- Pushes `(priority, ticket)` with no sequence number, which crashes with `TypeError` when two priorities tie (dicts cannot be compared).
- Sorts the whole list on every pop instead of using a heap.
- Uses `max` logic so priority 9 comes out before priority 1.
- Raises `IndexError` on an empty queue instead of returning `None` as asked.

In [ ]:
class TicketQueue:
    """Priority queue for tickets.

    Rules:
    - priority = ticket.get("priority", 5); lower number pops first
    - same priority pops in arrival order (FIFO)
    - pop() on an empty queue returns None
    """

    def __init__(self):
        pass

    def push(self, ticket: dict) -> None:
        raise NotImplementedError

    def pop(self) -> dict | None:
        raise NotImplementedError

In [ ]:
def replay_queue(queue_class, ops):
    """ops: ("push", ticket) or ("pop",). Returns the id of each popped ticket (None if empty)."""
    q = queue_class()
    popped = []
    for op in ops:
        if op[0] == "push":
            q.push(op[1])
        else:
            ticket = q.pop()
            popped.append(ticket["id"] if ticket else None)
    return popped

cases = [
    (([("push", {"id": "normal", "priority": 5}), ("push", {"id": "urgent", "priority": 1}),
       ("pop",), ("pop",)],), ["urgent", "normal"]),
    (([("push", {"id": "a", "priority": 3}), ("push", {"id": "b", "priority": 3}),
       ("push", {"id": "c", "priority": 3}), ("pop",), ("pop",), ("pop",)],), ["a", "b", "c"]),   # FIFO on ties
    (([("push", {"id": "x"}), ("push", {"id": "y", "priority": 1}), ("push", {"id": "z", "priority": 9}),
       ("pop",), ("pop",), ("pop",)],), ["y", "x", "z"]),                                         # missing = 5
    (([("pop",)],), [None]),
    (([("push", {"id": "a", "priority": 5}), ("pop",), ("push", {"id": "b", "priority": 5}),
       ("push", {"id": "c", "priority": 1}), ("pop",), ("pop",), ("pop",)],), ["a", "c", "b", None]),
]
run_checks(lambda *a: replay_queue(TicketQueue, *a), cases)

### Reference solution

Try first. Then compare.

In [ ]:
import heapq

class TicketQueueReference:
    def __init__(self):
        self._heap = []
        self._seq = 0

    def push(self, ticket: dict) -> None:
        heapq.heappush(self._heap, (ticket.get("priority", 5), self._seq, ticket))
        self._seq += 1

    def pop(self) -> dict | None:
        if not self._heap:
            return None
        return heapq.heappop(self._heap)[2]

assert run_checks(lambda *a: replay_queue(TicketQueueReference, *a), cases)

**Explain it in 30 seconds**

> "I use a heap, so the most urgent ticket is always on top. Each entry is priority, then a sequence number, then the ticket. The sequence number keeps tickets with the same priority in arrival order, and it also means Python never has to compare two ticket dicts. An empty queue returns None."

**Edge cases to mention**

- Two tickets with the same priority (order must be fair and repeatable).
- A ticket with no priority field.
- Popping from an empty queue.
- A flood of urgent tickets that starves normal ones.

**If they ask "how would this work in production?"**

- Use a durable queue (SQS, a database table, Kafka) so tickets survive a worker restart.
- Raise a ticket's priority as it waits (aging), so normal tickets are not stuck forever.
- Tie priority to the customer's support agreement, and send repeated failures to a dead-letter queue (Problem 36).

## Problem 40: Batch Processing Failure Summary

**The situation**

A nightly job embeds thousands of documents. In the morning the team needs a short summary: how many worked, which kinds of errors happened, and which documents to retry.

**Your task**

Write `summarize_batch(results)` returning `{"success": int, "failed": {error_label: count}, "failed_ids": [...]}`. Anything that is not `"success"` counts as failed. The error label is the text before the first `:`, trimmed and lowercased, or `"unknown"` if there is no error.

**Example**

`[{"id": "d1", "status": "failed", "error": "timeout"}, {"id": "d2", "status": "success"}]` gives `{"success": 1, "failed": {"timeout": 1}, "failed_ids": ["d1"]}`.

**Ask before coding**

- Which statuses exist? Is anything other than success a failure?
- Error messages often include details like times or ids. Should we group them by type?
- Do you want the failed document ids so they can be retried?

**Plan to say aloud**

1. Start a success count, a `Counter` for errors, and a list of failed ids.
2. For each result, if status is `"success"`, add one to success.
3. Otherwise, take the error text before `:`, trim and lowercase it, use `"unknown"` if empty, and count it.
4. Record the failed id if there is one.
5. Return the three parts in a dict.

### Prompt for your AI assistant

```text
Write a Python function summarize_batch(results: list[dict]) -> dict. Return {'success': int, 'failed': dict[str, int], 'failed_ids': list}. A result counts as success only if result.get('status') == 'success'; anything else (including a missing status) is a failure. Error label: (result.get('error') or ''), take the text before the first ':', strip and lowercase it; use 'unknown' if empty. Append result['id'] to failed_ids when present, in input order. Use collections.Counter and return a plain dict for 'failed'. Show only the function.
```

**Check the AI's code for**

- Only counts `status == "failed"`, so a missing or unexpected status disappears from the report.
- Uses the full error message as the label, so every timeout with a different duration becomes its own row.
- Crashes on `error: None` because it calls `.split` on `None`.
- Returns the `Counter` object itself; fine in Python, but it prints oddly and may not serialize as expected.

In [ ]:
def summarize_batch(results: list[dict]) -> dict:
    """Summarize a batch run.

    Rules:
    - status == "success" -> success count; anything else is a failure
    - error label = text before the first ":", stripped and lowercased; "unknown" if empty or missing
    - failed_ids lists failed result ids (when present), in input order
    - return {"success": int, "failed": {label: count}, "failed_ids": [...]}
    """
    raise NotImplementedError

In [ ]:
cases = [
    (([{"id": "d1", "status": "failed", "error": "timeout"}, {"id": "d2", "status": "success"}],),
     {"success": 1, "failed": {"timeout": 1}, "failed_ids": ["d1"]}),
    (([],), {"success": 0, "failed": {}, "failed_ids": []}),
    (([{"id": "a", "status": "failed", "error": "Timeout: took 31s"},
       {"id": "b", "status": "failed", "error": "timeout: took 45s"},
       {"id": "c", "status": "failed", "error": "rate_limited: 429"}],),
     {"success": 0, "failed": {"timeout": 2, "rate_limited": 1}, "failed_ids": ["a", "b", "c"]}),  # grouped by type
    (([{"id": "x", "status": "failed"}, {"id": "y"}, {"id": "z", "status": "failed", "error": None}],),
     {"success": 0, "failed": {"unknown": 3}, "failed_ids": ["x", "y", "z"]}),                      # missing data
    (([{"status": "skipped", "error": "empty file"}, {"id": "ok", "status": "success"}],),
     {"success": 1, "failed": {"empty file": 1}, "failed_ids": []}),                                 # no id to record
]
run_checks(summarize_batch, cases)

### Reference solution

Try first. Then compare.

In [ ]:
from collections import Counter

def summarize_batch_reference(results: list[dict]) -> dict:
    success = 0
    failed = Counter()
    failed_ids = []
    for result in results:
        if result.get("status") == "success":
            success += 1
            continue
        label = (result.get("error") or "").split(":")[0].strip().lower()
        failed[label or "unknown"] += 1
        if "id" in result:
            failed_ids.append(result["id"])
    return {"success": success, "failed": dict(failed), "failed_ids": failed_ids}

assert run_checks(summarize_batch_reference, cases)

**Explain it in 30 seconds**

> "I count successes, and treat anything else as a failure so nothing slips out of the report. Failures are grouped by error type, the part of the message before the colon, so a thousand timeouts show up as one row instead of a thousand. I also keep the failed ids so the team can retry exactly those documents."

**Edge cases to mention**

- Missing status or a status nobody expected, like `"skipped"`.
- Missing or `None` error message.
- Error messages with ids or timings that would create thousands of labels.
- An empty batch.

**If they ask "how would this work in production?"**

- Tag every run with a batch id and tenant id so summaries can be compared night to night.
- Split retryable errors (timeouts, rate limits) from permanent ones (corrupt file) and retry only the first kind.
- Open an incident automatically when the failure rate crosses a threshold, and show trends on a dashboard.

## Wrap-up

- Order and grouping must be predictable: a sequence number for ties, a fixed label rule for errors.
- Missing fields get a clear default (priority 5, error "unknown") instead of crashing or vanishing.
- The output should tell the team what to do next: which ticket is next, which documents to retry.